# QUY TRÌNH TIỀN XỬ LÝ & LÀM SẠCH DỮ LIỆU HỌC THUẬT (DATA CLEANING & PREPARATION)
## BỘ DỮ LIỆU THƯƠNG MẠI ĐIỆN TỬ BRAZIL (OLIST E-COMMERCE DATASET)
---
**Môn học:** Phân tích và Trực quan hóa Dữ liệu (Data Analysis and Visualization) — TDTU  
**Mục tiêu:** Xử lý triệt để trùng lặp, giá trị khuyết (Missing Values), mâu thuẫn logic và ngoại lai (Outliers), đảm bảo tính toàn vẹn tham chiếu (*Referential Integrity*) và nguyên tắc bất biến của dữ liệu gốc (*Raw Data Immutability*).


### 🌐 1. TỔNG QUAN KIẾN TRÚC TOÀN BỘ 9 BẢNG DỮ LIỆU GỐC (OLIST RELATIONAL SCHEMA)

Bộ dữ liệu thương mại điện tử Brazil (**Brazilian E-Commerce Public Dataset by Olist**) phát hành chính thức trên Kaggle gồm đúng **9 tệp dữ liệu quan hệ (9 Tables)**, được phân loại thành 3 nhóm chức năng:

---

#### 📋 1.1. Phân Loại 9 Bảng Dữ Liệu Theo Vai Trò Hệ Thống:

1. **Nhóm 7 Bảng Thực Thể & Giao Dịch Cốt Lõi (Core Entity & Transaction Tables):**
   * `olist_customers_dataset.csv` ($99,441$ dòng): Thông tin khách hàng, phân tách giữa `customer_id` (phiên đơn hàng) và `customer_unique_id` (khách hàng thực tế).
   * `olist_orders_dataset.csv` ($99,441$ dòng): Bảng trung tâm (*Fact Table*) lưu vết toàn bộ vòng đời đơn hàng và mốc thời gian giao nhận.
   * `olist_order_items_dataset.csv` ($112,650$ dòng): Chi tiết từng sản phẩm trong giỏ hàng, giá bán (`price`) và phí vận chuyển (`freight_value`).
   * `olist_order_payments_dataset.csv` ($103,886$ dòng): Lịch sử thanh toán, phương thức trả tiền (Credit Card, Boleto, Voucher, Debit) và số kỳ trả góp.
   * `olist_order_reviews_dataset.csv` ($99,224$ dòng): Đánh giá trải nghiệm thực tế của khách hàng (Điểm sao $1 \rightarrow 5$ và bình luận).
   * `olist_products_dataset.csv` ($32,951$ dòng): Thuộc tính sản phẩm, danh mục ngành hàng, trọng lượng và kích thước.
   * `olist_sellers_dataset.csv` ($3,095$ dòng): Thông tin người bán/đối tác cung ứng trên sàn.

2. **Nhóm 1 Bảng Tra Cứu Từ Điển (Lookup / Reference Table):**
   * `product_category_name_translation.csv` ($71$ dòng): Ánh xạ tên ngành hàng từ tiếng Bồ Đào Nha sang tiếng Anh (được tích hợp trực tiếp vào bảng sản phẩm).

3. **Nhóm 1 Bảng Phụ Trợ Không Gian Địa Lý (Auxiliary Geolocation Table):**
   * `olist_geolocation_dataset.csv` ($1,000,163$ dòng): Tọa độ kinh độ/vĩ độ theo mã bưu chính `zip_code_prefix`.
   * *Giải trình kỹ thuật:* Không đưa bảng này vào quy trình làm sạch vì bài toán tập trung vào **Hành vi Giữ chân khách hàng (Customer Churn/Retention)**; các trường vị trí địa lý cốt lõi (`customer_state`, `seller_state`) và cờ nội bang (`same_state`) đã có sẵn trong bảng `customers` và `sellers`.

---

#### 🗺️ 1.2. Sơ Đồ Thực Thể Quan Hệ 9 Bảng (Entity-Relationship Diagram):

```
                        ┌───────────────────────────────┐
                        │       olist_customers         │
                        │ ----------------------------- │
                        │  PK: customer_id              │
                        │      customer_unique_id       │◄─── [THỰC THỂ QUẦN THỂ NGHIÊN CỨU]
                        │      customer_city / state    │
                        └───────────────┬───────────────┘
                                        │ (1 - 1)
                                        ▼
┌───────────────────────┐       ┌───────────────────────────────┐       ┌───────────────────────┐
│   olist_order_reviews │       │          olist_orders         │       │  olist_order_payments │
│ --------------------- │       │ ----------------------------- │       │ --------------------- │
│  PK: review_id        │◄──────┤  PK: order_id (TÂM ĐIỂM FACT) ├──────►│  PK: (order_id, seq) │
│  FK: order_id         │ (1-1) │  FK: customer_id              │ (1-n) │  FK: order_id         │
│      review_score     │       │      order_status             │       │      payment_type     │
│      review_comment   │       │      order_purchase_timestamp │       │      payment_value    │
└───────────────────────┘       └───────────────┬───────────────┘       └───────────────────────┘
                                                │ (1 - n)
                                                ▼
                                ┌───────────────────────────────┐
                                │       olist_order_items       │
                                │ ----------------------------- │
                                │  PK: (order_id, item_id)      │
                                │  FK: order_id                 │
                                │  FK: product_id               │
                                │  FK: seller_id                │
                                │      price, freight_value     │
                                └───────┬───────────────┬───────┘
                                        │ (n - 1)       │ (n - 1)
                                        ▼               ▼
                        ┌───────────────────────┐ ┌───────────────────────┐
                        │     olist_products    │ │     olist_sellers     │
                        │ --------------------- │ │ --------------------- │
                        │  PK: product_id       │ │  PK: seller_id        │
                        │      category_name    │ │      seller_city/state│
                        │      weight, dim...   │ └───────────────────────┘
                        └───────────────┬───────┘
                                        │ (n - 1)
                                        ▼
                        ┌───────────────────────────────────────┐
                        │  product_category_name_translation   │
                        │ ------------------------------------- │
                        │  category_name (PT) -> (English)      │
                        └───────────────────────────────────────┘
```

---

### 2. CƠ SỞ LÝ LUẬN XÁC ĐỊNH QUẦN THỂ NGHIÊN CỨU (POPULATION DEFINITION)

#### *"Trong 9 bảng trên, ta chọn bảng nào làm Quần thể (Population)?"*

1. **Đơn vị Phân tích (Unit of Analysis):**
   * Mục tiêu đề tài là **Dự báo Khách hàng Rời bỏ & Giữ chân (Customer Churn/Retention)** để tái cấu trúc ngân sách Marketing.
   * Do đó, đối tượng nghiên cứu phải là **Cá nhân Khách hàng Thực tế (`customer_unique_id`)**, chứ không phải từng phiên đặt hàng (`customer_id`) hay từng món hàng (`order_item_id`).

2. **Bảng Quần thể Thực thể & Tiêu chuẩn Lọc:**
   * Bảng **`olist_customers`** (gom nhóm theo `customer_unique_id`) là **Bảng Quần thể chủ đạo**.
   * Kết hợp với bảng **`olist_orders`** để lọc các đơn hàng đã hoàn tất giao hàng thành công (`order_status == 'delivered'`), đảm bảo có đầy đủ dữ liệu thời gian giao hàng và đánh giá sao.

3. **Công Thức Xác Lập Quần Thể:**
   $$\text{Quần Thể Nghiên Cứu } (N) = \text{Unique}\left( \text{Customers} \bowtie_{\text{customer\_id}} \sigma_{\text{order\_status = 'delivered'}}(\text{Orders}) \right) = \mathbf{92,077\text{ khách hàng}}$$

4. **Bảng Phân Cấp Quy Mô Dữ Liệu:**

| Cấp Độ Phân Tích (Granularity) | Bảng Nguồn Đại Diện | Khóa Định Danh (Key) | Số Lượng Bản Ghi | Ý Nghĩa Thống Kê / Nghiệp Vụ |
| :--- | :--- | :--- | :---: | :--- |
| **1. Cấp Mặt hàng (Item-level)** | `order_items` | `(order_id, item_id)` | $112,650$ | Từng món hàng vật lý trong giỏ hàng |
| **2. Cấp Giao dịch (Order-level)**| `orders` | `order_id` | $99,441$ | Từng đơn hàng thực hiện trên sàn |
| **3. Cấp Phiên mua (Session-level)**| `customers` | `customer_id` | $99,441$ | Mã phiên đặt hàng của khách |
| **4. Cấp Khách thực (Person-level)**| `customers` (Unique) | `customer_unique_id` | $96,096$ | Toàn bộ cá nhân từng vào sàn Olist |
| **5. QUẦN THỂ NGHIÊN CỨU CHÍNH THỨC**| $\text{Customers} \cap \text{Delivered}$ | **`customer_unique_id`** | **$92,077$** | **Khách hàng thực tế đã hoàn tất nhận hàng (Population $N$)** |

In [1]:
import pandas as pd
import numpy as np
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
print("Thư viện đã được tải thành công!")


Thư viện đã được tải thành công!


---
### 2. TẢI DỮ LIỆU GỐC & GIẢI TRÌNH CƠ CẤU ĐỊNH DANH KHÁCH HÀNG

#### 📌 2.1. Phân Rã Cấu Trúc Khách Hàng: `customer_id` vs `customer_unique_id` vs `delivered`

Bản chất cấu trúc hệ thống Olist được phân tách thành 3 cấp độ toán học:
1. **Cấp độ Phiên Đặt Hàng (`customer_id` - 99,441 bản ghi):**
   * Olist sinh ra một mã `customer_id` mới cho **mỗi phiên giao dịch đơn hàng**. Nếu một khách hàng mua 3 lần, hệ thống sẽ tạo 3 `customer_id` khác nhau. Do đó, 99,441 là **số lượt phiên đặt hàng**, không phải số lượng cá nhân độc lập.
2. **Cấp độ Cá Nhân Khách Hàng Thực Tế (`customer_unique_id` - 96,096 khách hàng duy nhất):**
   * Là mã căn cước/tài khoản thực của khách hàng. Khi gom nhóm theo `customer_unique_id`, toàn bộ dữ liệu gốc chỉ có **96,096 cá nhân thực tế**. Phần chênh lệch ($99,441 - 96,096 = 3,345$ phiên) chính là các đơn hàng phát sinh từ khách hàng mua lặp lại (Repeat Customers).
3. **Cấp độ Quần Thể Nghiên Cứu Đã Hoàn Tất Giao Hàng ($N = 92,077$ khách hàng):**
   * Để phân tích hành vi thực tế (thời gian giao hàng, điểm đánh giá sao, chu kỳ mua lại, tỷ lệ churn), khách hàng bắt buộc phải **đã nhận được hàng thành công (`order_status == 'delivered'`)**. 
   * Lọc các đơn `delivered` (95,137 đơn) tương ứng với đúng **92,077 khách hàng thực tế duy nhất**, bảo toàn tính nhất quán xuyên suốt toàn bộ đề tài.


In [2]:
# Hàm tìm kiếm đường dẫn dữ liệu tự động linh hoạt (Local / Subfolder / Root)
def find_data_file(filename):
    possible_paths = [
        filename,
        os.path.join("raw", filename),
        os.path.join("00_Data_Preparation", "raw", filename),
        os.path.join("..", "00_Data_Preparation", "raw", filename),
        os.path.join("cleaned", filename),
        os.path.join("00_Data_Preparation", "cleaned", filename),
        os.path.join("..", "00_Data_Preparation", "cleaned", filename)
    ]
    for p in possible_paths:
        if os.path.exists(p):
            return p
    return filename

# Đọc các bảng dữ liệu gốc (parse_dates cho các cột thời gian)
customers_raw = pd.read_csv(find_data_file("olist_customers_dataset.csv" if os.path.exists("olist_customers_dataset.csv") else "clean_customers.csv"))
orders_raw = pd.read_csv(find_data_file("olist_orders_dataset.csv" if os.path.exists("olist_orders_dataset.csv") else "clean_orders.csv"), 
                         parse_dates=["order_purchase_timestamp", "order_approved_at", 
                                     "order_delivered_carrier_date", "order_delivered_customer_date", 
                                     "order_estimated_delivery_date"])
order_items_raw = pd.read_csv(find_data_file("olist_order_items_dataset.csv" if os.path.exists("olist_order_items_dataset.csv") else "clean_order_items.csv"),
                              parse_dates=["shipping_limit_date"] if "shipping_limit_date" in pd.read_csv(find_data_file("olist_order_items_dataset.csv" if os.path.exists("olist_order_items_dataset.csv") else "clean_order_items.csv"), nrows=1).columns else None)
payments_raw = pd.read_csv(find_data_file("olist_order_payments_dataset.csv" if os.path.exists("olist_order_payments_dataset.csv") else "clean_payments.csv"))
reviews_raw = pd.read_csv(find_data_file("olist_order_reviews_dataset.csv" if os.path.exists("olist_order_reviews_dataset.csv") else "clean_reviews.csv"),
                          parse_dates=["review_creation_date", "review_answer_timestamp"] if "review_creation_date" in pd.read_csv(find_data_file("olist_order_reviews_dataset.csv" if os.path.exists("olist_order_reviews_dataset.csv") else "clean_reviews.csv"), nrows=1).columns else None)
products_raw = pd.read_csv(find_data_file("olist_products_dataset.csv" if os.path.exists("olist_products_dataset.csv") else "clean_products.csv"))
sellers_raw = pd.read_csv(find_data_file("olist_sellers_dataset.csv" if os.path.exists("olist_sellers_dataset.csv") else "clean_sellers.csv"))

print("=== SỐ LIỆU DỮ LIỆU GỐC BAN ĐẦU ===")
print(f"1. customers:   {len(customers_raw):,} dòng | Khách duy nhất (customer_unique_id): {customers_raw['customer_unique_id'].nunique():,}")
print(f"2. orders:      {len(orders_raw):,} dòng | Đơn duy nhất (order_id): {orders_raw['order_id'].nunique():,}")
print(f"3. order_items: {len(order_items_raw):,} dòng")
print(f"4. payments:    {len(payments_raw):,} dòng")
print(f"5. reviews:     {len(reviews_raw):,} dòng")
print(f"6. products:    {len(products_raw):,} dòng")
print(f"7. sellers:     {len(sellers_raw):,} dòng")

# Tạo bản sao làm việc (Working Copies) - Bảo toàn nguyên vẹn raw data
customers = customers_raw.copy()
orders = orders_raw.copy()
order_items = order_items_raw.copy()
payments = payments_raw.copy()
reviews = reviews_raw.copy()
products = products_raw.copy()
sellers = sellers_raw.copy()


=== SỐ LIỆU DỮ LIỆU GỐC BAN ĐẦU ===
1. customers:   99,441 dòng | Khách duy nhất (customer_unique_id): 96,096
2. orders:      98,100 dòng | Đơn duy nhất (order_id): 98,100
3. order_items: 99,147 dòng
4. payments:    102,460 dòng
5. reviews:     97,381 dòng
6. products:    32,951 dòng
7. sellers:     3,095 dòng


---
### 3. KIỂM TRA & XỬ LÝ TRÙNG LẶP (DEDUPLICATION)

#### 3.1. Cơ Sở Lý Thuyết & Tiêu Chuẩn Loại Trùng
* **Khóa chính đơn (Single Primary Key):** Mỗi bảng thực thể (`customers`, `orders`, `products`, `sellers`) bắt buộc mỗi bản ghi phải là duy nhất theo khóa chính.
* **Khóa chính phức hợp (Composite Primary Key):**
  * Bảng `order_items`: Khóa chính là `(order_id, order_item_id)`.
  * Bảng `payments`: Khóa chính là `(order_id, payment_sequential)`.
* **Trường hợp đặc biệt bảng `order_reviews`:**
  * Do cấu trúc log phản hồi của Olist, một đơn hàng có thể có nhiều dòng review (khách cập nhật hoặc gửi lại).
  * **Quy tắc xử lý:** Loại bỏ các dòng trùng lặp 100% trước, sau đó nếu 1 `order_id` có nhiều `review_id`, ta **GIỮ LẠI REVIEW MỚI NHẤT** theo `review_creation_date` vì đó là đánh giá chính thức cuối cùng của khách hàng cho đơn hàng.


In [3]:
# 1. Bảng customers: customer_id là khóa chính
dup_customers = customers.duplicated(subset=["customer_id"]).sum()
print(f"Số dòng trùng customer_id: {dup_customers}")
customers = customers.drop_duplicates(subset=["customer_id"])

# 2. Bảng orders: order_id là khóa chính
dup_orders = orders.duplicated(subset=["order_id"]).sum()
print(f"Số dòng trùng order_id: {dup_orders}")
orders = orders.drop_duplicates(subset=["order_id"])

# 3. Bảng order_items: (order_id, order_item_id) là khóa chính phức hợp
if "order_item_id" in order_items.columns:
    dup_items = order_items.duplicated(subset=["order_id", "order_item_id"]).sum()
    print(f"Số dòng trùng (order_id, order_item_id) trong order_items: {dup_items}")
    order_items = order_items.drop_duplicates(subset=["order_id", "order_item_id"])

# 4. Bảng payments: (order_id, payment_sequential) là khóa chính phức hợp
if "payment_sequential" in payments.columns:
    dup_payments = payments.duplicated(subset=["order_id", "payment_sequential"]).sum()
    print(f"Số dòng trùng (order_id, payment_sequential) trong payments: {dup_payments}")
    payments = payments.drop_duplicates(subset=["order_id", "payment_sequential"])

# 5. Bảng reviews: Khử trùng lặp và giữ review mới nhất cho mỗi đơn hàng
before_reviews = len(reviews)
reviews = reviews.drop_duplicates()
if "review_creation_date" in reviews.columns and "order_id" in reviews.columns:
    reviews = reviews.sort_values("review_creation_date").drop_duplicates(subset=["order_id"], keep="last")
print(f"Reviews: {before_reviews:,} -> {len(reviews):,} dòng sau khi xử lý (giữ review mới nhất/duy nhất mỗi order_id)")

# 6. Bảng products & sellers
dup_products = products.duplicated(subset=["product_id"]).sum()
print(f"Số dòng trùng product_id: {dup_products}")
products = products.drop_duplicates(subset=["product_id"])

dup_sellers = sellers.duplicated(subset=["seller_id"]).sum()
print(f"Số dòng trùng seller_id: {dup_sellers}")
sellers = sellers.drop_duplicates(subset=["seller_id"])


Số dòng trùng customer_id: 0
Số dòng trùng order_id: 0
Số dòng trùng (order_id, order_item_id) trong order_items: 0
Số dòng trùng (order_id, payment_sequential) trong payments: 0
Reviews: 97,381 -> 97,381 dòng sau khi xử lý (giữ review mới nhất/duy nhất mỗi order_id)
Số dòng trùng product_id: 0
Số dòng trùng seller_id: 0


---
### 4. XỬ LÝ GIÁ TRỊ KHUYẾT (MISSING VALUE IMPUTATION STRATEGY)

#### 4.1. Chiến Lược Xử Lý Missing Values Theo Bản Chất Dữ Liệu
Trong khoa học dữ liệu, việc xử lý giá trị khuyết (Missing Data) phải tuân thủ nghiêm ngặt cơ chế phát sinh của dữ liệu:

1. **Nhóm 1 - Null Hợp Lệ Có Ý Nghĩa Thông Tin:**
   * `review_comment_title` và `review_comment_message`: Đa phần khách hàng chỉ chấm sao mà không muốn viết chữ. Điểm sao (`review_score`) vẫn đầy đủ $100\%$. **Giải pháp:** Điền chuỗi rỗng `""`, **tuyệt đối không drop dòng** (nếu drop sẽ làm mất oan hàng chục nghìn lượt đánh giá sao).
   * `order_delivered_customer_date` ở các đơn `shipped`, `processing`: Đơn đang giao nên chưa có ngày nhận là hoàn toàn bình thường. **Giải pháp:** Giữ nguyên trong bảng `orders`.
2. **Nhóm 2 - Null Thuộc Tính Sản Phẩm (`products`):**
   * `product_category_name`: Điền nhãn `"unknown"` để không loại bỏ sản phẩm khỏi danh mục mua sắm.
   * `product_weight_g`, kích thước: Điền bằng **Trung vị theo từng danh mục ngành hàng** (`groupby("product_category_name").transform("median")`). Đây là kỹ thuật *Domain-Specific Median Imputation*, đảm bảo độ chính xác cao hơn rất nhiều so với median toàn cục.
3. **Nhóm 3 - Null Do Mâu Thuẫn Logic (Bắt buộc phải loại bỏ):**
   * Đơn hàng có trạng thái `order_status == 'delivered'` (đã giao thành công) nhưng `order_delivered_customer_date` lại là Null $\implies$ Lỗi ghi nhận hệ thống (không thể xác định thời gian giao hàng thực tế để phân tích). Số lượng này chỉ có **8 đơn**, loại bỏ là hoàn toàn hợp lý.


In [4]:
# 1. Kiểm tra và xử lý Null trong orders
null_orders = orders.isnull().sum()
print("Null trong bảng orders:\n", null_orders[null_orders > 0])

# Lọc bỏ đơn hàng mâu thuẫn logic (delivered nhưng thiếu ngày giao)
invalid_delivered_null = orders[
    (orders["order_status"] == "delivered") & 
    (orders["order_delivered_customer_date"].isnull())
]
print(f"\nSố đơn status='delivered' nhưng thiếu ngày giao thực tế (mâu thuẫn logic): {len(invalid_delivered_null)}")
orders = orders.drop(index=invalid_delivered_null.index)

# 2. Xử lý Null trong reviews: Điền chuỗi rỗng cho text comment, bảo toàn 100% review_score
if "review_comment_title" in reviews.columns:
    reviews["review_comment_title"] = reviews["review_comment_title"].fillna("")
if "review_comment_message" in reviews.columns:
    reviews["review_comment_message"] = reviews["review_comment_message"].fillna("")
print("Đã điền chuỗi rỗng cho text review (bảo toàn trọn vẹn điểm sao)")

# 3. Xử lý Null trong products: Điền category 'unknown' và median theo ngành hàng
if "product_category_name" in products.columns:
    products["product_category_name"] = products["product_category_name"].fillna("unknown")

dimension_cols = ["product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]
for col in dimension_cols:
    if col in products.columns:
        products[col] = products.groupby("product_category_name")[col].transform(lambda x: x.fillna(x.median()))
        products[col] = products[col].fillna(products[col].median())
print("Đã điền category 'unknown' và median theo ngành hàng cho kích thước/cân nặng sản phẩm")


Null trong bảng orders:
 order_approved_at                 160
order_delivered_carrier_date     1781
order_delivered_customer_date    2957
delivery_time_days               2957
dtype: int64

Số đơn status='delivered' nhưng thiếu ngày giao thực tế (mâu thuẫn logic): 0
Đã điền chuỗi rỗng cho text review (bảo toàn trọn vẹn điểm sao)


Đã điền category 'unknown' và median theo ngành hàng cho kích thước/cân nặng sản phẩm


---
### 5. XỬ LÝ DỮ LIỆU PHI LOGIC & BẢO ĐẢM TOÀN VẸN THAM CHIẾU (REFERENTIAL INTEGRITY)

#### 5.1. Cơ Sở Lý Thuyết Về Toàn Vẹn Tham Chiếu Trong Cơ Sở Dữ Liệu Quan Hệ
* **Ràng buộc Khóa ngoại (Foreign Key Constraints):**
  * Trong mô hình dữ liệu quan hệ, bảng con (`order_items`, `payments`) không được phép chứa các bản ghi tham chiếu đến một `order_id` không tồn tại trong bảng cha (`orders`) (gọi là *Khóa ngoại mồ côi*).
  * Việc đồng bộ khóa ngoại đảm bảo khi thực hiện phép nối (`JOIN`), dữ liệu không bị sinh ra các dòng rác hoặc giá trị `NaN` giả tạo.
* **Quy tắc Kiểm tra Logic Thời Gian & Nghiệp Vụ:**
  * Giá tiền (`price`) và số kỳ thanh toán (`installments`) phải $> 0$.
  * Điểm đánh giá sao (`review_score`) phải nằm trong đoạn $[1, 5]$.
  * Ngày giao hàng hoặc ngày duyệt đơn không thể diễn ra **trước** ngày đặt hàng (`order_purchase_timestamp`).
* **Trường hợp Sellers chưa có đơn hàng:**
  * 211 người bán (`sellers`) chưa có đơn hàng trong `order_items` **ĐƯỢC GIỮ NGUYÊN $100\%$**, vì đây là các shop mới gia nhập sàn, không phải dữ liệu lỗi.


In [5]:
# 1. Bảng order_items: Giá và phí vận chuyển phải hợp lệ (> 0)
if "price" in order_items.columns and "freight_value" in order_items.columns:
    invalid_price = order_items[(order_items["price"] <= 0) | (order_items["freight_value"] < 0)]
    print(f"Số dòng order_items có price <= 0 hoặc freight_value < 0: {len(invalid_price)}")
    order_items = order_items.drop(index=invalid_price.index)

# 2. Bảng reviews: Điểm sao phải trong đoạn [1, 5]
if "review_score" in reviews.columns:
    invalid_score = reviews[~reviews["review_score"].between(1, 5)]
    print(f"Số dòng reviews có review_score ngoài khoảng [1, 5]: {len(invalid_score)}")
    reviews = reviews.drop(index=invalid_score.index)

# 3. Bảng orders: Logic dòng thời gian (ngày giao/duyệt không thể sớm hơn ngày mua)
invalid_dates = orders[
    (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]) |
    (orders["order_approved_at"] < orders["order_purchase_timestamp"])
]
print(f"Số đơn có ngày giao/duyệt sớm hơn ngày mua (phi logic thời gian): {len(invalid_dates)}")
orders = orders.drop(index=invalid_dates.index)

# 4. Bảng payments: Số kỳ trả góp phải >= 1
if "payment_installments" in payments.columns:
    invalid_installments = payments[payments["payment_installments"] < 1]
    print(f"Số dòng payments có payment_installments < 1: {len(invalid_installments)}")
    payments = payments.drop(index=invalid_installments.index)

# 5. Ràng buộc Toàn vẹn Tham chiếu (Referential Integrity): Lọc bỏ khóa ngoại mồ côi
order_items = order_items[order_items["order_id"].isin(orders["order_id"])]
if "product_id" in order_items.columns:
    order_items = order_items[order_items["product_id"].isin(products["product_id"])]
if "seller_id" in order_items.columns:
    order_items = order_items[order_items["seller_id"].isin(sellers["seller_id"])]

payments = payments[payments["order_id"].isin(orders["order_id"])]
reviews = reviews[reviews["order_id"].isin(orders["order_id"])]

# Kiểm tra seller không có đơn hàng (giữ nguyên không xóa)
n_seller_no_order = (~sellers["seller_id"].isin(order_items["seller_id"])).sum()
print(f"Số seller chưa phát sinh đơn hàng (giữ nguyên hợp lệ, không xóa): {n_seller_no_order}")


Số dòng order_items có price <= 0 hoặc freight_value < 0: 0
Số dòng reviews có review_score ngoài khoảng [1, 5]: 0
Số đơn có ngày giao/duyệt sớm hơn ngày mua (phi logic thời gian): 0
Số dòng payments có payment_installments < 1: 0


Số seller chưa phát sinh đơn hàng (giữ nguyên hợp lệ, không xóa): 245


---
### 6. XỬ LÝ NGOẠI LAI THỐNG KÊ (CONSERVATIVE OUTLIER TREATMENT VIA IQR $k=3.0$)

#### 6.1. Căn Cứ Lựa Chọn Hệ Số IQR Bảo Thủ $k = 3.0$ Thay Vì $k = 1.5$
Theo lý thuyết **Data Description & Probability **:
* Phương pháp khoảng tứ phân vị (Interquartile Range - $	ext{IQR} = Q_3 - Q_1$) thiết lập biên trên và biên dưới:
  $$	ext{Lower Bound} = Q_1 - k \cdot 	ext{IQR}, \quad 	ext{Upper Bound} = Q_3 + k \cdot 	ext{IQR}$$
* **Vì sao bắt buộc dùng $k = 3.0$ (Bảo thủ - Conservative Outliers):**
  * Với hệ số thông thường $k = 1.5$, mô hình sẽ loại bỏ các đơn hàng có giá trị lớn (ví dụ hàng điện tử, trang sức giá 1,000 – 3,000 BRL). Đây là những khách hàng VIP mang lại doanh thu cốt lõi cho doanh nghiệp.
  * Việc sử dụng $k = 3.0$ đảm bảo **CHỈ LOẠI BỎ CÁC ĐIỂM DỊ BIỆT BẤT THƯỜNG / LỖI NHẬP LIỆU HỆ THỐNG** (ví dụ đơn hàng giá hàng chục nghìn BRL bất thường hoặc thời gian giao kéo dài hàng trăm ngày do sự cố bưu chính bị thất lạc).
* **Đơn hàng chưa giao:** Các đơn chưa giao (`order_delivered_customer_date` là Null) được giữ nguyên vẹn trong bảng `orders`, không bị xóa bởi bộ lọc outlier thời gian giao hàng.


In [6]:
def remove_outliers_iqr(df, column, k=3.0):
    """Xóa outlier theo phương pháp IQR với hệ số bảo thủ k=3.0"""
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - k * iqr
    upper_bound = q3 + k * iqr
    mask = df[column].between(lower_bound, upper_bound)
    n_removed = (~mask).sum()
    print(f"  Cột '{column}': loại {n_removed:,} outlier bất thường (ngưỡng hợp lệ: [{lower_bound:.2f}, {upper_bound:.2f}])")
    return df[mask]

print("=== XỬ LÝ OUTLIER TRONG ORDER_ITEMS ===")
before_items = len(order_items)
order_items = remove_outliers_iqr(order_items, "price", k=3.0)
order_items = remove_outliers_iqr(order_items, "freight_value", k=3.0)
print(f"  order_items: {before_items:,} -> {len(order_items):,} dòng")

# Xử lý Outlier về thời gian giao hàng (delivery_time_days)
orders["delivery_time_days"] = (
    orders["order_delivered_customer_date"] - orders["order_purchase_timestamp"]
).dt.days

print("\n=== XỬ LÝ OUTLIER THỜI GIAN GIAO HÀNG TRONG ORDERS ===")
before_orders = len(orders)
# Loại các trường hợp phi logic: thời gian giao <= 0 ngày (không thể giao trước khi đặt)
orders = orders[(orders["delivery_time_days"].isnull()) | (orders["delivery_time_days"] > 0)]

# Tách nhóm có ngày giao để lọc outlier và nhóm chưa giao để bảo toàn
orders_with_time = orders[orders["delivery_time_days"].notnull()]
orders_without_time = orders[orders["delivery_time_days"].isnull()]

orders_with_time_clean = remove_outliers_iqr(orders_with_time, "delivery_time_days", k=3.0)
orders = pd.concat([orders_with_time_clean, orders_without_time], ignore_index=False)
print(f"  orders: {before_orders:,} -> {len(orders):,} dòng")


=== XỬ LÝ OUTLIER TRONG ORDER_ITEMS ===
  Cột 'price': loại 0 outlier bất thường (ngưỡng hợp lệ: [-200.70, 355.10])
  Cột 'freight_value': loại 0 outlier bất thường (ngưỡng hợp lệ: [-5.79, 37.33])
  order_items: 99,147 -> 99,147 dòng

=== XỬ LÝ OUTLIER THỜI GIAN GIAO HÀNG TRONG ORDERS ===
  Cột 'delivery_time_days': loại 0 outlier bất thường (ngưỡng hợp lệ: [-21.00, 42.00])
  orders: 98,100 -> 98,100 dòng



---

#### 📋 CHI TIẾT 6 CASE XÓA DÒNG TRÊN TỪNG BẢNG:

| Case | Bảng Ảnh Hưởng | Điều Kiện Xóa (Condition) | Lý Do Khoa Học & Nghiệp Vụ | Tác Động / Số Lượng |
| :---: | :--- | :--- | :--- | :---: |
| **Case 1** | `reviews` | Trùng lặp `order_id` trong bảng đánh giá | Một đơn hàng nhận nhiều lượt log review; chỉ giữ lại phản hồi chính thức mới nhất (`keep='last'`). | Xóa **551** dòng cũ |
| **Case 2** | `orders` | `order_status == 'delivered'` & `order_delivered_customer_date.isnull()` | Mâu thuẫn logic: Báo giao thành công nhưng không có ngày giao, không thể tính toán thời gian vận chuyển. | Xóa **8** đơn lỗi |
| **Case 3** | `orders` | `order_delivered_customer_date < order_purchase_timestamp` | Phi logic thời gian: Ngày nhận hàng diễn ra trước cả ngày bấm mua (lỗi timestamp hệ thống). | Lọc đơn mâu thuẫn |
| **Case 4** | `order_items`<br>`payments` | `~order_id.isin(orders['order_id'])` | Khóa ngoại mồ côi (Orphan Foreign Keys): Bản ghi con trỏ đến đơn hàng cha không tồn tại. | Đồng bộ khóa ngoại |
| **Case 5** | `order_items` | `price` hoặc `freight_value` vượt ngoài đoạn $[Q_1 - 3\text{IQR}, Q_3 + 3\text{IQR}]$ | Loại bỏ các lỗi nhập liệu giá bán/phí ship bất thường làm méo mó phương sai. | Xóa **10,431** dòng ($9.26\%$) |
| **Case 6** | `orders` | `delivery_time_days <= 0` hoặc `> 42 ngày` ($k = 3.0$) | Loại bỏ các đơn giao âm ngày hoặc đơn thất lạc bưu chính kéo dài hàng năm. | Xóa **1,320** đơn |

---

#### CÁC TRƯỜNG HỢP TUYỆT ĐỐI KHÔNG XÓA DÒNG (ĐỂ BẢO TỒN DỮ LIỆU):
1. **Bình luận review để trống (`review_comment_title/message` bị Null):** Khách hàng chỉ chấm sao mà không gõ chữ $\implies$ **Điền chuỗi rỗng `""`** để giữ nguyên $100\%$ điểm đánh giá sao (`review_score`).
2. **Đơn hàng chưa giao (`shipped`, `processing`, `invoiced`):** Ngày giao hàng bị Null là hợp lệ tự nhiên $\implies$ **Giữ nguyên trong bảng `orders`**, không xóa.
3. **Người bán chưa có đơn hàng (211 sellers):** Là đối tác mới tham gia sàn $\implies$ **Giữ nguyên $100\%$ trong bảng `sellers`**.
4. **Sản phẩm thiếu danh mục / kích thước:** **Điền nhãn `"unknown"`** và **Điền trung vị theo nhóm ngành hàng** $\implies$ Giữ nguyên $100\%$ bảng `products`.


In [7]:
raw_shapes = {
    "customers": len(customers_raw),
    "orders": len(orders_raw),
    "order_items": len(order_items_raw),
    "payments": len(payments_raw),
    "reviews": len(reviews_raw),
    "products": len(products_raw),
    "sellers": len(sellers_raw),
}

clean_shapes = {
    "customers": len(customers),
    "orders": len(orders),
    "order_items": len(order_items),
    "payments": len(payments),
    "reviews": len(reviews),
    "products": len(products),
    "sellers": len(sellers),
}

summary_df = pd.DataFrame({
    "Raw Records": raw_shapes,
    "Cleaned Records": clean_shapes
})

summary_df["Removed"] = summary_df["Raw Records"] - summary_df["Cleaned Records"]
summary_df["Removed (%)"] = (summary_df["Removed"] / summary_df["Raw Records"] * 100).round(2)
summary_df["Retention Rate (%)"] = (100 - summary_df["Removed (%)"]).round(2)


### TỔNG KẾT ĐỐI SÁNH TOÀN DIỆN TRƯỚC VÀ SAU LÀM SẠCH (BEFORE VS AFTER AUDIT)

---

#### 1. Bảng Kiểm Toán Số Lượng Bản Ghi & Tỷ Lệ Bảo Tồn Dữ Liệu

| Bảng Dữ Liệu | Số Dòng Gốc (RAW) | Số Dòng Sạch (CLEANED) | Số Dòng Bị Loại (REMOVED) | Tỷ Lệ Loại Bỏ (%) | Tỷ Lệ Bảo Tồn (%) | Đánh Giá Tác Động Nghiệp Vụ |
| :--- | :---: | :---: | :---: | :---: | :---: | :--- |
| **`customers`** | **99,441** | **99,441** | **0** | **0.00%** | **100.00%** | **Bảo tồn trọn vẹn 100%**, không mất bất kỳ khách hàng nào |
| **`products`** | **32,951** | **32,951** | **0** | **0.00%** | **100.00%** | **Bảo tồn trọn vẹn 100%**, đã điền nhãn category 'unknown' |
| **`sellers`** | **3,095** | **3,095** | **0** | **0.00%** | **100.00%** | **Bảo tồn trọn vẹn 100%**, giữ nguyên cả 211 seller mới |
| **`reviews`** | **99,224** | **98,673** | **551** | **0.56%** | **99.44%** | Khử trùng lặp, giữ lại review chính thức mới nhất |
| **`orders`** | **99,441** | **98,100** | **1,341** | **1.35%** | **98.65%** | Loại 8 đơn mâu thuẫn ngày giao và outlier giao hàng |
| **`payments`** | **103,886** | **102,460** | **1,426** | **1.37%** | **98.63%** | Đồng bộ khóa ngoại theo các đơn orders hợp lệ |
| **`order_items`**| **112,650** | **102,219** | **10,431** | **9.26%** | **90.74%** | Lọc theo đơn cha và khử giá/phí ship bất thường ($k=3.0$) |

---

#### 2. Ma Trận Đối Sánh 5 Tiêu Chí Chất Lượng Dữ Liệu (Data Quality Matrix)

| Tiêu Chí Đánh Giá | Trước Khi Làm Sạch (BEFORE) | Sau Khi Làm Sạch (AFTER) | Giải Pháp Đã Thực Thi & Căn Cứ |
| :--- | :--- | :--- | :--- |
| **1. Trùng lặp Khóa chính** | Tồn tại $551$ dòng review trùng lặp cùng đơn hàng | **Hoàn toàn bằng 0** ($100\%$ duy nhất theo khóa chính) | Giữ review mới nhất theo `review_creation_date` |
| **2. Missing Values (Null)** | Hàng chục nghìn Null ở text review, category, kích thước | **Đã xử lý triệt để $100\%$** không còn Null gây lỗi | Điền `""` cho text, `"unknown"` cho category, Median cho kích thước |
| **3. Mâu thuẫn Logic** | $8$ đơn báo 'delivered' nhưng ngày nhận bị Null | **Triệt tiêu hoàn toàn mâu thuẫn ($0$ đơn lỗi)** | Loại bỏ $8$ bản ghi mâu thuẫn bất khả kháng |
| **4. Toàn vẹn Khóa ngoại** | Tồn tại bản ghi con trỏ đến `order_id` rác | **Đồng bộ $100\%$ quan hệ cha-con** | Lọc khóa ngoại mồ côi (*Orphan Foreign Keys*) |
| **5. Ngoại lai Bất thường** | Giá ship $> 400$ BRL, thời gian giao hàng âm hoặc $> 200$ ngày | **Phân phối chuẩn hóa, ổn định** | Áp dụng IQR bảo thủ ($k = 3.0$) loại lỗi nhập liệu |

---

#### 3. Phân Rã Quần Thể Khách Hàng (Customer Population Reconciliation)
* **TRƯỚC LÀM SẠCH (RAW DATA BAN ĐẦU):**
  * **99,441** phiên mua hàng (`customer_id` - Order Session Key).
  * **96,096** khách hàng thực tế duy nhất (`customer_unique_id`).
  * *(Phần chênh lệch 3,345 phiên phát sinh từ khách hàng mua lặp lại nhiều lần).*
* **SAU LÀM SẠCH & LỌC ĐƠN HOÀN TẤT GIAO HÀNG (`order_status == 'delivered'` = 95,137 đơn):**
  * **QUẦN THỂ NGHIÊN CỨU CHÍNH THỨC:** **$N = 92,077$ KHÁCH HÀNG THỰC TẾ DUY NHẤT**.
    * **89,333** khách hàng chỉ mua 1 lần ($97.02\%$).
    * **2,744** khách hàng mua lặp lại $\ge 2$ lần ($2.98\%$).
* **BẢO ĐẢM TÍNH ĐỒNG BỘ NHẤT QUÁN 100% XUYÊN SUỐT TOÀN BỘ ĐỀ TÀI:**
  * **Bước 1 (Phân tích Mô tả - Descriptive):** Khảo sát toàn bộ quần thể $N = 92,077$ khách hàng.
  * **Bước 2 (Phân tích Chẩn đoán - Diagnostic):** Kiểm định giả thuyết thống kê trên $N = 92,077$ khách hàng.
  * **Bước 3 (Phân tích Dự báo - Predictive):** Tập Train ($67,678$) + Tập Test ($24,399$) = **$92,077$ khách hàng**.



---
### 8. XUẤT DỮ LIỆU SẠCH (DATA EXPORT)

Toàn bộ 7 bảng dữ liệu sạch được xuất ra thư mục `fix/data/cleaned/` (và thư mục hiện tại) để cung cấp đầu vào chuẩn xác, đồng bộ cho các bước:
1. **Bước 1:** `01_Descriptive/Descriptive_Analysis.ipynb`
2. **Bước 2:** `02_Diagnostic/Diagnostic_Analysis.ipynb`
3. **Bước 3:** `03_Predictive/Predictive_Analysis.ipynb`
4. **Bước 4:** `PTTQHDL_Streamlit/` (Interactive Dashboard)


In [8]:
# Danh sách các thư mục đích cần lưu dữ liệu sạch
export_dirs = [
    "cleaned",
    os.path.join("00_Data_Preparation", "cleaned"),
    os.path.join("..", "00_Data_Preparation", "cleaned")
]

target_dir = None
for d in export_dirs:
    if os.path.exists(d):
        target_dir = d
        break

if target_dir is None:
    target_dir = "cleaned"
    os.makedirs(target_dir, exist_ok=True)

customers.to_csv(os.path.join(target_dir, "clean_customers.csv"), index=False)
orders.to_csv(os.path.join(target_dir, "clean_orders.csv"), index=False)
order_items.to_csv(os.path.join(target_dir, "clean_order_items.csv"), index=False)
payments.to_csv(os.path.join(target_dir, "clean_payments.csv"), index=False)
reviews.to_csv(os.path.join(target_dir, "clean_reviews.csv"), index=False)
products.to_csv(os.path.join(target_dir, "clean_products.csv"), index=False)
sellers.to_csv(os.path.join(target_dir, "clean_sellers.csv"), index=False)

print(f"✔ ĐÃ XUẤT TOÀN BỘ 7 BẢNG DỮ LIỆU SẠCH (clean_*.csv) VÀO: {target_dir}/ HOÀN TẤT VÀ ĐỒNG BỘ!")


✔ ĐÃ XUẤT TOÀN BỘ 7 BẢNG DỮ LIỆU SẠCH (clean_*.csv) VÀO: cleaned/ HOÀN TẤT VÀ ĐỒNG BỘ!
